# CHC (CHIRPS) — daily rainfall quickstart (live)

Download a few days of **CHIRPS-2.0 daily precipitation** for a small area over Colombia, then read and plot the cropped GeoTIFF. CHC uses **anonymous FTP**, so no credentials are needed.

> This notebook performs a real network download from `data.chc.ucsb.edu`.

## Setup

Imports plus a bit of log-suppression. `earthlens` provides the unified `EarthLens` entry point; `pyramids` provides `Dataset` for reading and plotting the GeoTIFF later. We disable the `earthlens` logger for a clean notebook.

In [ ]:
import tempfile

from loguru import logger

from earthlens.core import EarthLens

logger.disable('earthlens')  # quieten the download logs for a clean notebook

## Build the request

Configure the CHC backend: the `daily` CHIRPS-2.0 product, a three-day window in January 2009, the `precipitation` variable, and a small bounding box over Colombia. Output GeoTIFFs go to a fresh temporary directory.

In [ ]:
lens = EarthLens(
    data_source='chc',
    temporal_resolution='daily',
    start='2009-01-01',
    end='2009-01-03',
    variables=['precipitation'],
    lat_lim=[4.19, 4.64],
    lon_lim=[-75.65, -74.73],
    path=tempfile.mkdtemp(prefix='chirps_'),
)

## Download the daily tiles

`download()` fetches each global daily tile, crops it to the requested bbox, and returns the list of written GeoTIFF paths.

In [ ]:
paths = lens.download(progress_bar=False)  # returns the written GeoTIFF paths
print(f'{len(paths)} files written')

The backend writes one `global-daily_precipitation_<date>.tif` per day.

In [ ]:
for p in sorted(paths):
    print(p.name)

## Read and plot the cropped raster

Read the first day's GeoTIFF with `pyramids` and pull its array. CHIRPS encodes no-data as `-9999`, so we mask negatives to `NaN` before plotting.

In [ ]:
import numpy as np
from pyramids.dataset import Dataset

ds = Dataset.read_file(sorted(paths)[0])
arr = np.asarray(ds.read_array(), dtype='float32')
arr = arr[0] if arr.ndim == 3 else arr
arr = np.where(arr < 0, np.nan, arr)  # CHIRPS no-data is -9999
print('EPSG', ds.epsg, '| shape', arr.shape)

Plot the masked precipitation field for the first day.

In [ ]:
glyph = ds.plot(cmap='Blues', title='CHIRPS daily rainfall — 2009-01-01')
glyph.cbar.set_label('precipitation (mm/day)')